# Managing Windows Updates and Direct Server Auto-Restart

This notebook addresses two important concerns when using Windows 11 Home as a development server without Docker:

1. **Controlling Windows Update behavior** to minimize unexpected restarts
2. **Configuring automatic server restarts** for directly-running services after system reboots

While Windows 11 Home doesn't allow completely disabling updates (unlike Pro/Enterprise editions), we can still implement strategies to better manage them and ensure our development environment recovers automatically after reboots.


## 1. Managing Windows Update Behavior

### Checking Current Windows Update Settings

First, let's check your current Windows Update settings using PowerShell commands. These commands will help you understand your current configuration.

In [ ]:
# Check current Windows Update settings
Get-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "ActiveHoursStart" -ErrorAction SilentlyContinue
Get-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "ActiveHoursEnd" -ErrorAction SilentlyContinue

# Check if updates are currently paused
$updateSettings = Get-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -ErrorAction SilentlyContinue
if ($updateSettings.PausedFeatureStatus -eq 1) {
    Write-Host "Feature updates are paused until: $($updateSettings.PausedFeatureDate)"
} else {
    Write-Host "Feature updates are not paused"
}

if ($updateSettings.PausedQualityStatus -eq 1) {
    Write-Host "Quality updates are paused until: $($updateSettings.PausedQualityDate)"
} else {
    Write-Host "Quality updates are not paused"
}

### Options for Managing Windows Update Restarts

While Windows 11 Home doesn't allow completely disabling updates, here are the most effective approaches you can use:

#### Option 1: Configure Active Hours

Setting active hours tells Windows not to restart during times when you're actively using your computer (up to an 18-hour window).

In [ ]:
# Set active hours (adjust to your working hours)
# Note: Windows 11 allows up to 18 hours between start and end
# This example sets active hours from 8:00 AM to 2:00 AM (18 hours)
$startHour = 8  # 8:00 AM
$endHour = 2    # 2:00 AM next day

try {
    # You need to run PowerShell as Administrator for this to work
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "ActiveHoursStart" -Value $startHour -Type DWord
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "ActiveHoursEnd" -Value $endHour -Type DWord
    Write-Host "Successfully set active hours from $startHour:00 to $endHour:00"
} catch {
    Write-Host "Error setting active hours. Make sure you're running PowerShell as Administrator."
    Write-Host "You can also set this via Windows Settings > Windows Update > Advanced options > Active hours"
}

#### Option 2: Pause Updates Temporarily

You can pause updates for a limited time (up to 35 days in Windows 11 Home). This is useful when you need guaranteed uptime for a specific period.

In [ ]:
# Pause updates for the maximum allowed time (usually 5 weeks/35 days)
# Note: This must be run as Administrator to work
try {
    # Calculate the date 35 days from today
    $pauseUntil = (Get-Date).AddDays(35).ToString("yyyy-MM-dd")
    
    # Set registry keys to pause updates
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "PausedFeatureStatus" -Value 1 -Type DWord
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "PausedFeatureDate" -Value $pauseUntil
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "PausedQualityStatus" -Value 1 -Type DWord
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Microsoft\WindowsUpdate\UX\Settings" -Name "PausedQualityDate" -Value $pauseUntil
    
    Write-Host "Updates paused until $pauseUntil"
} catch {
    Write-Host "Error pausing updates. Make sure you're running PowerShell as Administrator."
    Write-Host "You can also pause updates via Windows Settings > Windows Update > Pause updates"
}

#### Option 3: Configure Update Notification Settings

You can modify the Registry to get more notifications before updates. While this doesn't prevent updates, it gives you more warning time.

In [ ]:
# Configure Windows Update to notify before downloading and installing
# Note: Must run as Administrator
try {
    # This increases the notification window for pending restarts
    New-Item -Path "HKLM:\SOFTWARE\Policies\Microsoft\Windows\WindowsUpdate" -Force | Out-Null
    New-Item -Path "HKLM:\SOFTWARE\Policies\Microsoft\Windows\WindowsUpdate\AU" -Force | Out-Null
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Policies\Microsoft\Windows\WindowsUpdate\AU" -Name "AUOptions" -Value 2 -Type DWord
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Policies\Microsoft\Windows\WindowsUpdate" -Name "SetAutoRestartNotificationDisable" -Value 0 -Type DWord
    Set-ItemProperty -Path "HKLM:\SOFTWARE\Policies\Microsoft\Windows\WindowsUpdate" -Name "NoAutoRebootWithLoggedOnUsers" -Value 1 -Type DWord
    
    Write-Host "Update notification settings applied successfully"
} catch {
    Write-Host "Error configuring update settings. Make sure you're running PowerShell as Administrator."
}

#### Option 4: Use the Group Policy Editor (not available in Windows 11 Home)

For reference only: Windows 11 Pro/Enterprise/Education users have access to the Group Policy Editor which provides more control over update behavior. Home edition users would need to upgrade to gain access to these features.

#### Recommendation

The most reliable approach is to **combine options 1 and 2** - set appropriate active hours AND pause updates during critical periods when your server absolutely cannot be interrupted.

## 2. Creating Direct Server Startup Scripts for FastAPI and Vite

Now let's create scripts to directly start your FastAPI backend and Vite frontend servers, without using Docker.

In [ ]:
# File: start_backend.py
# This script starts the FastAPI backend server directly

import os
import sys
import subprocess
import logging
from datetime import datetime
from pathlib import Path
import time

# Configure logging
log_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "backend_startup.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def start_backend_server():
    """Start the FastAPI backend server directly"""
    try:
        # Record startup attempt
        logging.info(f"Starting backend server at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Change to project directory
        backend_dir = "C:/Users/USER/Documents/workspace/partnersclub/simulation/src/backend"
        os.chdir(backend_dir)
        logging.info(f"Changed directory to {backend_dir}")
        
        # Check if the server is already running on port 8000
        try:
            result = subprocess.run(
                ["powershell", "-Command", "Get-NetTCPConnection -LocalPort 8000 -ErrorAction SilentlyContinue"],
                capture_output=True,
                text=True,
                check=False
            )
            
            if "LocalPort" in result.stdout:
                logging.info("Backend server already running on port 8000")
                return True
        except Exception as e:
            logging.warning(f"Error checking if port 8000 is in use: {e}")
        
        # Start the backend server
        logging.info("Starting FastAPI server...")
        
        # Use subprocess.Popen to start as a background process
        with open(log_dir / "backend_output.log", "a") as log_file:
            process = subprocess.Popen(
                ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
                stdout=log_file,
                stderr=log_file,
                # Use detached process creation flags to ensure it continues running
                creationflags=subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP
            )
        
        # Give it a moment to start
        time.sleep(5)
        
        # Check if the process is running
        if process.poll() is None:  # None means it's still running
            logging.info(f"Backend server started with PID {process.pid}")
            
            # Save PID to file for later reference
            with open(log_dir / "backend.pid", "w") as f:
                f.write(str(process.pid))
            
            return True
        else:
            logging.error(f"Backend server failed to start, exit code: {process.returncode}")
            return False
    
    except Exception as e:
        logging.error(f"Error starting backend server: {e}")
        return False

if __name__ == "__main__":
    success = start_backend_server()
    sys.exit(0 if success else 1)

In [ ]:
# File: start_frontend.py
# This script starts the Vite frontend server directly

import os
import sys
import subprocess
import logging
from datetime import datetime
from pathlib import Path
import time

# Configure logging
log_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "frontend_startup.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def start_frontend_server():
    """Start the Vite frontend server directly"""
    try:
        # Record startup attempt
        logging.info(f"Starting frontend server at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Change to project directory
        frontend_dir = "C:/Users/USER/Documents/workspace/partnersclub/simulation/src/my-pwa-frontend"
        os.chdir(frontend_dir)
        logging.info(f"Changed directory to {frontend_dir}")
        
        # Check if the server is already running on port 5173
        try:
            result = subprocess.run(
                ["powershell", "-Command", "Get-NetTCPConnection -LocalPort 5173 -ErrorAction SilentlyContinue"],
                capture_output=True,
                text=True,
                check=False
            )
            
            if "LocalPort" in result.stdout:
                logging.info("Frontend server already running on port 5173")
                return True
        except Exception as e:
            logging.warning(f"Error checking if port 5173 is in use: {e}")
        
        # Install dependencies if needed (only if node_modules doesn't exist)
        if not os.path.isdir("node_modules"):
            logging.info("Installing frontend dependencies...")
            try:
                subprocess.run(["npm", "ci"], check=True)
            except subprocess.CalledProcessError as e:
                logging.error(f"Failed to install dependencies: {e}")
                return False
        
        # Start the frontend server
        logging.info("Starting Vite frontend server...")
        
        # Use subprocess.Popen to start as a background process
        with open(log_dir / "frontend_output.log", "a") as log_file:
            process = subprocess.Popen(
                ["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", "5173"],
                stdout=log_file,
                stderr=log_file,
                # Use detached process creation flags to ensure it continues running
                creationflags=subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP
            )
        
        # Give it a moment to start
        time.sleep(10)
        
        # Check if the process is running
        if process.poll() is None:  # None means it's still running
            logging.info(f"Frontend server started with PID {process.pid}")
            
            # Save PID to file for later reference
            with open(log_dir / "frontend.pid", "w") as f:
                f.write(str(process.pid))
            
            return True
        else:
            logging.error(f"Frontend server failed to start, exit code: {process.returncode}")
            return False
    
    except Exception as e:
        logging.error(f"Error starting frontend server: {e}")
        return False

if __name__ == "__main__":
    success = start_frontend_server()
    sys.exit(0 if success else 1)

## 3. Creating a Combined Startup Script

Let's create a combined script that starts both the frontend and backend servers:

In [ ]:
# File: start_servers.py
# This script starts both frontend and backend servers

import os
import sys
import logging
import time
from datetime import datetime
from pathlib import Path
import importlib.util

# Configure logging
log_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "server_startup.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def import_script(script_path):
    """Import a Python script as a module"""
    spec = importlib.util.spec_from_file_location("module.name", script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def start_all_servers():
    """Start both backend and frontend servers"""
    try:
        project_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation")
        
        # Import the server scripts
        backend_script_path = project_dir / "start_backend.py"
        frontend_script_path = project_dir / "start_frontend.py"
        
        # Create the scripts if they don't exist
        if not backend_script_path.exists():
            logging.info("Creating backend startup script...")
            with open(backend_script_path, "w") as f:
                f.write("""
import os
import sys
import subprocess
import logging
from datetime import datetime
from pathlib import Path
import time

# Configure logging
log_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "backend_startup.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def start_backend_server():
    \"\"\"Start the FastAPI backend server directly\"\"\"
    try:
        # Record startup attempt
        logging.info(f"Starting backend server at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Change to project directory
        backend_dir = "C:/Users/USER/Documents/workspace/partnersclub/simulation/src/backend"
        os.chdir(backend_dir)
        logging.info(f"Changed directory to {backend_dir}")
        
        # Check if the server is already running on port 8000
        try:
            result = subprocess.run(
                ["powershell", "-Command", "Get-NetTCPConnection -LocalPort 8000 -ErrorAction SilentlyContinue"],
                capture_output=True,
                text=True,
                check=False
            )
            
            if "LocalPort" in result.stdout:
                logging.info("Backend server already running on port 8000")
                return True
        except Exception as e:
            logging.warning(f"Error checking if port 8000 is in use: {e}")
        
        # Start the backend server
        logging.info("Starting FastAPI server...")
        
        # Use subprocess.Popen to start as a background process
        with open(log_dir / "backend_output.log", "a") as log_file:
            process = subprocess.Popen(
                ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
                stdout=log_file,
                stderr=log_file,
                # Use detached process creation flags to ensure it continues running
                creationflags=subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP
            )
        
        # Give it a moment to start
        time.sleep(5)
        
        # Check if the process is running
        if process.poll() is None:  # None means it's still running
            logging.info(f"Backend server started with PID {process.pid}")
            
            # Save PID to file for later reference
            with open(log_dir / "backend.pid", "w") as f:
                f.write(str(process.pid))
            
            return True
        else:
            logging.error(f"Backend server failed to start, exit code: {process.returncode}")
            return False
    
    except Exception as e:
        logging.error(f"Error starting backend server: {e}")
        return False

if __name__ == "__main__":
    success = start_backend_server()
    sys.exit(0 if success else 1)
""")

        if not frontend_script_path.exists():
            logging.info("Creating frontend startup script...")
            with open(frontend_script_path, "w") as f:
                f.write("""
import os
import sys
import subprocess
import logging
from datetime import datetime
from pathlib import Path
import time

# Configure logging
log_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "frontend_startup.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def start_frontend_server():
    \"\"\"Start the Vite frontend server directly\"\"\"
    try:
        # Record startup attempt
        logging.info(f"Starting frontend server at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Change to project directory
        frontend_dir = "C:/Users/USER/Documents/workspace/partnersclub/simulation/src/my-pwa-frontend"
        os.chdir(frontend_dir)
        logging.info(f"Changed directory to {frontend_dir}")
        
        # Check if the server is already running on port 5173
        try:
            result = subprocess.run(
                ["powershell", "-Command", "Get-NetTCPConnection -LocalPort 5173 -ErrorAction SilentlyContinue"],
                capture_output=True,
                text=True,
                check=False
            )
            
            if "LocalPort" in result.stdout:
                logging.info("Frontend server already running on port 5173")
                return True
        except Exception as e:
            logging.warning(f"Error checking if port 5173 is in use: {e}")
        
        # Install dependencies if needed (only if node_modules doesn't exist)
        if not os.path.isdir("node_modules"):
            logging.info("Installing frontend dependencies...")
            try:
                subprocess.run(["npm", "ci"], check=True)
            except subprocess.CalledProcessError as e:
                logging.error(f"Failed to install dependencies: {e}")
                return False
        
        # Start the frontend server
        logging.info("Starting Vite frontend server...")
        
        # Use subprocess.Popen to start as a background process
        with open(log_dir / "frontend_output.log", "a") as log_file:
            process = subprocess.Popen(
                ["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", "5173"],
                stdout=log_file,
                stderr=log_file,
                # Use detached process creation flags to ensure it continues running
                creationflags=subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP
            )
        
        # Give it a moment to start
        time.sleep(10)
        
        # Check if the process is running
        if process.poll() is None:  # None means it's still running
            logging.info(f"Frontend server started with PID {process.pid}")
            
            # Save PID to file for later reference
            with open(log_dir / "frontend.pid", "w") as f:
                f.write(str(process.pid))
            
            return True
        else:
            logging.error(f"Frontend server failed to start, exit code: {process.returncode}")
            return False
    
    except Exception as e:
        logging.error(f"Error starting frontend server: {e}")
        return False

if __name__ == "__main__":
    success = start_frontend_server()
    sys.exit(0 if success else 1)
""")

        # Now import and run both scripts
        backend_module = import_script(backend_script_path)
        frontend_module = import_script(frontend_script_path)
        
        # Start backend first
        logging.info("Starting backend server...")
        backend_success = backend_module.start_backend_server()
        
        if backend_success:
            logging.info("Backend server started successfully")
        else:
            logging.error("Failed to start backend server")
            return False
        
        # Then start frontend
        logging.info("Starting frontend server...")
        frontend_success = frontend_module.start_frontend_server()
        
        if frontend_success:
            logging.info("Frontend server started successfully")
        else:
            logging.error("Failed to start frontend server")
            return False
        
        logging.info("All servers started successfully!")
        return True
        
    except Exception as e:
        logging.error(f"Error in start_all_servers: {e}")
        return False

if __name__ == "__main__":
    success = start_all_servers()
    sys.exit(0 if success else 1)

## 4. Creating Windows Scheduled Tasks for Automatic Startup

Now, let's create a script to set up a Windows Task Scheduler task that will run our startup script when the system boots:

In [ ]:
# File: setup_autostart_task.py
# This script creates a scheduled task to start our servers when Windows boots

import os
import subprocess
import sys
from pathlib import Path

def create_startup_task():
    """Create a Windows scheduled task to run our start script at system startup"""
    # Get the current directory where our scripts are located
    script_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation")
    start_script = script_dir / "start_servers.py"
    
    # Make sure the script exists before creating the task
    if not start_script.exists():
        print(f"Error: Start script not found at {start_script}")
        print("Please run the notebook cell that creates this script first")
        return False
    
    # Task name
    task_name = "SimulationServersAutoStart"
    python_path = sys.executable
    command = f'"{python_path}" "{start_script}"'
    
    # Create task XML file
    task_xml = f"""<?xml version="1.0" encoding="UTF-16"?>
<Task version="1.2" xmlns="http://schemas.microsoft.com/windows/2004/02/mit/task">
  <RegistrationInfo>
    <Description>Start simulation servers at system startup</Description>
  </RegistrationInfo>
  <Triggers>
    <BootTrigger>
      <Enabled>true</Enabled>
      <Delay>PT1M</Delay> <!-- 1 minute delay after startup -->
    </BootTrigger>
  </Triggers>
  <Principals>
    <Principal id="Author">
      <LogonType>InteractiveToken</LogonType>
      <RunLevel>HighestAvailable</RunLevel>
    </Principal>
  </Principals>
  <Settings>
    <MultipleInstancesPolicy>IgnoreNew</MultipleInstancesPolicy>
    <DisallowStartIfOnBatteries>false</DisallowStartIfOnBatteries>
    <StopIfGoingOnBatteries>false</StopIfGoingOnBatteries>
    <AllowHardTerminate>true</AllowHardTerminate>
    <StartWhenAvailable>true</StartWhenAvailable>
    <RunOnlyIfNetworkAvailable>false</RunOnlyIfNetworkAvailable>
    <IdleSettings>
      <StopOnIdleEnd>false</StopOnIdleEnd>
      <RestartOnIdle>false</RestartOnIdle>
    </IdleSettings>
    <AllowStartOnDemand>true</AllowStartOnDemand>
    <Enabled>true</Enabled>
    <Hidden>false</Hidden>
    <RunOnlyIfIdle>false</RunOnlyIfIdle>
    <WakeToRun>false</WakeToRun>
    <ExecutionTimeLimit>PT0S</ExecutionTimeLimit> <!-- No time limit -->
    <Priority>7</Priority>
  </Settings>
  <Actions Context="Author">
    <Exec>
      <Command>{python_path}</Command>
      <Arguments>"{start_script}"</Arguments>
      <WorkingDirectory>{script_dir}</WorkingDirectory>
    </Exec>
  </Actions>
</Task>
"""
    
    # Write the XML to a temporary file
    xml_path = script_dir / "task.xml"
    with open(xml_path, "w", encoding="utf-16") as f:
        f.write(task_xml)
    
    try:
        # Create the task using the XML file
        subprocess.run(["schtasks", "/create", "/tn", task_name, "/xml", str(xml_path), "/f"], check=True)
        print(f"Successfully created startup task '{task_name}'")
        print(f"The servers will start automatically after system boot")
        
        # Clean up
        os.remove(xml_path)
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error creating task: {e}")
        print("You may need to run this script as Administrator")
        return False

if __name__ == "__main__":
    success = create_startup_task()
    sys.exit(0 if success else 1)

## 5. Server Status Checker

Let's create a script to verify that our servers are running properly:

In [ ]:
# File: check_server_status.py
# This script checks if our servers are running properly

import subprocess
import requests
import logging
import sys
from pathlib import Path
import os
import psutil

# Configure logging
log_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_dir / "server_status.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

def check_process_by_pid_file(pid_file):
    """Check if a process is running by reading its PID from a file"""
    try:
        if not os.path.exists(pid_file):
            return False
        
        with open(pid_file, "r") as f:
            pid = int(f.read().strip())
        
        # Check if the process exists
        return psutil.pid_exists(pid)
    except Exception as e:
        logging.error(f"Error checking PID file {pid_file}: {e}")
        return False

def check_port_in_use(port):
    """Check if a port is in use"""
    try:
        result = subprocess.run(
            ["powershell", "-Command", f"Get-NetTCPConnection -LocalPort {port} -ErrorAction SilentlyContinue"],
            capture_output=True,
            text=True,
            check=False
        )
        return "LocalPort" in result.stdout
    except Exception as e:
        logging.error(f"Error checking port {port}: {e}")
        return False

def check_backend_api():
    """Check if our backend API is responding"""
    try:
        # Check PID file
        backend_pid_file = log_dir / "backend.pid"
        pid_exists = check_process_by_pid_file(backend_pid_file)
        
        # Check port
        port_in_use = check_port_in_use(8000)
        
        # Check health endpoint
        try:
            response = requests.get("http://localhost:8000/health", timeout=5)
            api_healthy = response.status_code == 200
        except Exception:
            api_healthy = False
            
        if pid_exists and port_in_use and api_healthy:
            logging.info("✅ Backend API is fully operational")
            return True
        else:
            status = []
            if not pid_exists:
                status.append("Process not running")
            if not port_in_use:
                status.append("Port 8000 not in use")
            if not api_healthy:
                status.append("Health check failed")
                
            logging.warning(f"❌ Backend API issues: {', '.join(status)}")
            return False
            
    except Exception as e:
        logging.error(f"❌ Error checking backend API: {e}")
        return False

def check_frontend():
    """Check if frontend is serving"""
    try:
        # Check PID file
        frontend_pid_file = log_dir / "frontend.pid"
        pid_exists = check_process_by_pid_file(frontend_pid_file)
        
        # Check port
        port_in_use = check_port_in_use(5173)
        
        # Try to access frontend
        try:
            response = requests.get("http://localhost:5173", timeout=5)
            serving_content = response.status_code == 200
        except Exception:
            serving_content = False
            
        if pid_exists and port_in_use and serving_content:
            logging.info("✅ Frontend is fully operational")
            return True
        else:
            status = []
            if not pid_exists:
                status.append("Process not running")
            if not port_in_use:
                status.append("Port 5173 not in use")
            if not serving_content:
                status.append("Not serving content")
                
            logging.warning(f"❌ Frontend issues: {', '.join(status)}")
            return False
            
    except Exception as e:
        logging.error(f"❌ Error checking frontend: {e}")
        return False

def restart_backend_if_needed():
    """Restart the backend server if it's not running"""
    if not check_backend_api():
        logging.info("Attempting to restart backend server...")
        script_path = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/start_backend.py")
        if script_path.exists():
            try:
                subprocess.run([sys.executable, str(script_path)], check=True)
                logging.info("Backend restart initiated")
            except Exception as e:
                logging.error(f"Failed to restart backend: {e}")
        else:
            logging.error(f"Backend startup script not found at {script_path}")

def restart_frontend_if_needed():
    """Restart the frontend server if it's not running"""
    if not check_frontend():
        logging.info("Attempting to restart frontend server...")
        script_path = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation/start_frontend.py")
        if script_path.exists():
            try:
                subprocess.run([sys.executable, str(script_path)], check=True)
                logging.info("Frontend restart initiated")
            except Exception as e:
                logging.error(f"Failed to restart frontend: {e}")
        else:
            logging.error(f"Frontend startup script not found at {script_path}")

if __name__ == "__main__":
    logging.info("Starting server status check...")
    
    backend_ok = check_backend_api()
    frontend_ok = check_frontend()
    
    if "--auto-restart" in sys.argv:
        if not backend_ok:
            restart_backend_if_needed()
        
        if not frontend_ok:
            restart_frontend_if_needed()
    
    if backend_ok and frontend_ok:
        logging.info("✅ All systems operational")
        sys.exit(0)
    else:
        logging.warning("❌ Some services are not running correctly")
        sys.exit(1)

## 6. Creating a Monitoring Task

Let's create a scheduled task that regularly checks if our servers are running and restarts them if necessary:

In [ ]:
# File: setup_monitoring_task.py
# This script creates a scheduled task to periodically monitor server status

import subprocess
import sys
from pathlib import Path

def create_monitoring_task():
    """Create a Windows scheduled task for regular monitoring"""
    # Script paths
    script_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation")
    monitor_script = script_dir / "check_server_status.py"
    
    # Make sure the script exists
    if not monitor_script.exists():
        print(f"Error: Monitor script not found at {monitor_script}")
        print("Please run the notebook cell that creates this script first")
        return False
    
    task_name = "SimulationServerMonitor"
    python_path = sys.executable
    command = f'"{python_path}" "{monitor_script}" --auto-restart'
    
    # Task XML for running every 15 minutes
    task_xml = f"""<?xml version="1.0" encoding="UTF-16"?>
<Task version="1.2" xmlns="http://schemas.microsoft.com/windows/2004/02/mit/task">
  <RegistrationInfo>
    <Description>Monitor simulation servers and restart if needed</Description>
  </RegistrationInfo>
  <Triggers>
    <TimeTrigger>
      <Repetition>
        <Interval>PT15M</Interval> <!-- Every 15 minutes -->
        <StopAtDurationEnd>false</StopAtDurationEnd>
      </Repetition>
      <StartBoundary>2025-01-01T00:00:00</StartBoundary>
      <Enabled>true</Enabled>
    </TimeTrigger>
  </Triggers>
  <Principals>
    <Principal id="Author">
      <LogonType>InteractiveToken</LogonType>
      <RunLevel>HighestAvailable</RunLevel>
    </Principal>
  </Principals>
  <Settings>
    <MultipleInstancesPolicy>IgnoreNew</MultipleInstancesPolicy>
    <DisallowStartIfOnBatteries>false</DisallowStartIfOnBatteries>
    <StopIfGoingOnBatteries>false</StopIfGoingOnBatteries>
    <AllowHardTerminate>true</AllowHardTerminate>
    <StartWhenAvailable>true</StartWhenAvailable>
    <RunOnlyIfNetworkAvailable>false</RunOnlyIfNetworkAvailable>
    <IdleSettings>
      <StopOnIdleEnd>false</StopOnIdleEnd>
      <RestartOnIdle>false</RestartOnIdle>
    </IdleSettings>
    <AllowStartOnDemand>true</AllowStartOnDemand>
    <Enabled>true</Enabled>
    <Hidden>false</Hidden>
    <RunOnlyIfIdle>false</RunOnlyIfIdle>
    <WakeToRun>false</WakeToRun>
    <ExecutionTimeLimit>PT5M</ExecutionTimeLimit> <!-- 5 minute time limit -->
    <Priority>7</Priority>
  </Settings>
  <Actions Context="Author">
    <Exec>
      <Command>{python_path}</Command>
      <Arguments>"{monitor_script}" --auto-restart</Arguments>
      <WorkingDirectory>{script_dir}</WorkingDirectory>
    </Exec>
  </Actions>
</Task>
"""
    
    # Write the XML to a temporary file
    xml_path = script_dir / "monitor_task.xml"
    with open(xml_path, "w", encoding="utf-16") as f:
        f.write(task_xml)
    
    try:
        # Create the task using the XML file
        subprocess.run(["schtasks", "/create", "/tn", task_name, "/xml", str(xml_path), "/f"], check=True)
        print(f"Successfully created monitoring task '{task_name}'")
        print(f"The servers will be checked every 15 minutes and restarted if needed")
        
        # Clean up
        import os
        os.remove(xml_path)
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error creating task: {e}")
        print("You may need to run this script as Administrator")
        return False

if __name__ == "__main__":
    success = create_monitoring_task()
    sys.exit(0 if success else 1)

## 7. Setting Up Everything with a Single Script

Let's create a main setup script that you can run once to set up everything:

In [ ]:
# File: setup_all.py
# This script sets up all components of the auto-restart system

import os
import sys
import subprocess
from pathlib import Path
import importlib.util

def import_script(script_path):
    """Import a Python script as a module"""
    spec = importlib.util.spec_from_file_location("module.name", script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def main():
    """Set up all components of the auto-restart system"""
    project_dir = Path("C:/Users/USER/Documents/workspace/partnersclub/simulation")
    
    # Create logs directory
    log_dir = project_dir / "logs"
    log_dir.mkdir(exist_ok=True)
    
    # Define script paths
    start_servers_script = project_dir / "start_servers.py"
    start_backend_script = project_dir / "start_backend.py"
    start_frontend_script = project_dir / "start_frontend.py"
    check_status_script = project_dir / "check_server_status.py"
    setup_autostart_script = project_dir / "setup_autostart_task.py"
    setup_monitoring_script = project_dir / "setup_monitoring_task.py"
    
    # Check if we're running as Administrator
    is_admin = False
    try:
        result = subprocess.run(
            ["powershell", "-Command", "([Security.Principal.WindowsPrincipal] [Security.Principal.WindowsIdentity]::GetCurrent()).IsInRole([Security.Principal.WindowsBuiltInRole]::Administrator)"],
            capture_output=True,
            text=True,
            check=True
        )
        is_admin = result.stdout.strip().lower() == "true"
    except Exception:
        pass
    
    if not is_admin:
        print("WARNING: This script should be run as Administrator to create scheduled tasks")
        print("Some operations might fail without administrator privileges")
    
    # Import and run each setup component
    try:
        # 1. Create and run each script
        print("\n1. Creating server startup scripts...")
        
        # Create the scripts if they don't exist
        if not start_backend_script.exists():
            print(f"Creating {start_backend_script}...")
            module_code = importlib.util.find_spec("backend_start_script", [str(project_dir)]).loader.get_code()
            with open(start_backend_script, "w") as f:
                f.write(module_code)
        
        if not start_frontend_script.exists():
            print(f"Creating {start_frontend_script}...")
            module_code = importlib.util.find_spec("frontend_start_script", [str(project_dir)]).loader.get_code()
            with open(start_frontend_script, "w") as f:
                f.write(module_code)
        
        if not start_servers_script.exists():
            print(f"Creating {start_servers_script}...")
            module_code = importlib.util.find_spec("servers_start_script", [str(project_dir)]).loader.get_code()
            with open(start_servers_script, "w") as f:
                f.write(module_code)
        
        if not check_status_script.exists():
            print(f"Creating {check_status_script}...")
            module_code = importlib.util.find_spec("check_status_script", [str(project_dir)]).loader.get_code()
            with open(check_status_script, "w") as f:
                f.write(module_code)
        
        # 2. Create scheduled tasks
        print("\n2. Creating scheduled tasks...")
        
        # Create autostart task
        if not setup_autostart_script.exists():
            print(f"Creating {setup_autostart_script}...")
            module_code = importlib.util.find_spec("autostart_task_script", [str(project_dir)]).loader.get_code()
            with open(setup_autostart_script, "w") as f:
                f.write(module_code)
        
        print("Setting up autostart task...")
        autostart_module = import_script(setup_autostart_script)
        autostart_result = autostart_module.create_startup_task()
        
        # Create monitoring task
        if not setup_monitoring_script.exists():
            print(f"Creating {setup_monitoring_script}...")
            module_code = importlib.util.find_spec("monitoring_task_script", [str(project_dir)]).loader.get_code()
            with open(setup_monitoring_script, "w") as f:
                f.write(module_code)
        
        print("Setting up monitoring task...")
        monitoring_module = import_script(setup_monitoring_script)
        monitoring_result = monitoring_module.create_monitoring_task()
        
        # 3. Start the servers now
        print("\n3. Starting servers now...")
        servers_module = import_script(start_servers_script)
        servers_result = servers_module.start_all_servers()
        
        # 4. Check status
        print("\n4. Checking server status...")
        status_module = import_script(check_status_script)
        backend_ok = status_module.check_backend_api()
        frontend_ok = status_module.check_frontend()
        
        # 5. Summary
        print("\n=== SETUP SUMMARY ===")
        print(f"Autostart task created: {'✓' if autostart_result else '✗'}")
        print(f"Monitoring task created: {'✓' if monitoring_result else '✗'}")
        print(f"Servers started: {'✓' if servers_result else '✗'}")
        print(f"Backend status: {'✓' if backend_ok else '✗'}")
        print(f"Frontend status: {'✓' if frontend_ok else '✗'}")
        
        print("\nSetup complete!")
        print("Your servers will now start automatically after Windows reboots.")
        print("They will also be monitored every 15 minutes and restarted if they stop working.")
        
        if not is_admin:
            print("\nNOTE: You ran this script without Administrator privileges.")
            print("To fully set up the scheduled tasks, run this script as Administrator.")
        
    except Exception as e:
        print(f"Error during setup: {e}")
        return 1
    
    return 0

if __name__ == "__main__":
    sys.exit(main())

## Summary and Recommendations

### Managing Windows Updates

1. **Set Active Hours** (8:00 AM to 2:00 AM) to prevent updates during your working hours
2. **Pause Updates** during critical periods when you need guaranteed uptime
3. **Configure Update Notifications** to get more warning before restarts

### Server Auto-Restart (Without Docker)

1. I've provided scripts to directly start your:
   - **FastAPI backend** on port 8000 using uvicorn
   - **Vite frontend** on port 5173 using npm run dev

2. These scripts automatically:
   - Check if servers are already running
   - Install dependencies if needed
   - Start the servers as detached processes
   - Save the process IDs for monitoring

3. **Windows Task Scheduler Integration**:
   - One task starts your servers at system boot
   - Another task checks every 15 minutes if servers are still running

### Complete Setup Instructions

1. **Save all the scripts** from this notebook to your project directory
2. **Run the `setup_all.py` script as Administrator**:
   - Open PowerShell as Administrator
   - Run: `python C:\Users\USER\Documents\workspace\partnersclub\simulation\setup_all.py`
3. **Verify everything works** by:
   - Checking the logs directory
   - Accessing your frontend at http://localhost:5173
   - Accessing your backend API at http://localhost:8000

After completing these steps, your servers will automatically restart whenever Windows reboots, ensuring your development environment is always available.